In [1]:
# ============================================================
# CELL 1: Tokenizer + Data
# ============================================================
import os, random, math, json, pickle
import numpy as np, pandas as pd
import sentencepiece as spm
import torch
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

WORKDIR = '/kaggle/working'
os.chdir(WORKDIR)
print("Working dir:", os.getcwd())

# --- Load + clean ---
base = "/kaggle/input/datasets/siddhanttimalsina/nepali-text-summarizer/"
df = pd.concat([
    pd.read_csv(base+"small.csv"),
    pd.read_csv(base+"medium.csv"),
    pd.read_csv(base+"large.csv"),
], ignore_index=True)
df = df.dropna(subset=['content','summary'])
df = df.drop_duplicates(subset=['content'])
df['content'] = df['content'].astype(str).str.strip()
df['summary'] = df['summary'].astype(str).str.strip()
df = df[df['content'].str.len()>10]
df = df[df['summary'].str.len()>5]
print("Rows:", len(df))

# --- Corpus ---
with open("corpus.txt", "w", encoding="utf-8") as f:
    for t in df['content']: f.write(t.replace("\n"," ")+"\n")
    for t in df['summary']: f.write(t.replace("\n"," ")+"\n")

# --- Train BPE SentencePiece ---
spm.SentencePieceTrainer.train(
    input="corpus.txt", model_prefix="nepali_sp",
    vocab_size=32000, model_type="bpe", character_coverage=1.0,
    max_sentence_length=100000,
    pad_id=0, unk_id=1, bos_id=2, eos_id=3,
    split_by_whitespace=True, split_by_unicode_script=True,
    num_threads=os.cpu_count(), shuffle_input_sentence=True)

sp = spm.SentencePieceProcessor()
sp.load("nepali_sp.model")
PAD_ID, BOS_ID, EOS_ID = sp.pad_id(), sp.bos_id(), sp.eos_id()
VOCAB_SIZE = sp.get_piece_size()
print(f"Tokenizer: vocab={VOCAB_SIZE}, PAD={PAD_ID}, BOS={BOS_ID}, EOS={EOS_ID}")

# --- Encode ---
MAX_SRC_LEN, MAX_TGT_LEN = 400, 64
def tok_src(t): return sp.encode(str(t), out_type=int)[:MAX_SRC_LEN]
def tok_tgt(t):
    ids = sp.encode(str(t), out_type=int)[:MAX_TGT_LEN-2]
    return [BOS_ID]+ids+[EOS_ID]
df['src_ids'] = df['content'].apply(tok_src)
df['tgt_ids'] = df['summary'].apply(tok_tgt)

train_df, temp_df = train_test_split(df, test_size=0.2, random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=SEED)
print(f"Train {len(train_df)} | Val {len(val_df)} | Test {len(test_df)}")

# Persist
with open("splits.pkl", "wb") as f:
    pickle.dump({'train': train_df, 'val': val_df, 'test': test_df}, f)
print("Saved nepali_sp.model, corpus.txt, splits.pkl")

Working dir: /kaggle/working
Rows: 49914


I0000 00:00:1794541456.561806      23 sentencepiece_trainer.cc:105] Starts training with : 
trainer_spec {
  input: corpus.txt
  input_format: 
  model_prefix: nepali_sp
  model_type: BPE
  vocab_size: 32000
  self_test_sample_size: 0
  character_coverage: 1
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 100000
  num_threads: 4
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 1
  bos_id: 2
  eos_id: 3
  pad_id: 0
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  ⁇ 
  enable_differentia

Tokenizer: vocab=32000, PAD=0, BOS=2, EOS=3
Train 39931 | Val 4991 | Test 4992
Saved nepali_sp.model, corpus.txt, splits.pkl


In [2]:
# ============================================================
# CELL 2: Model + Training
# ============================================================
import os, json, math, time
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

class SummarizationDataset(Dataset):
    def __init__(self, frame):
        self.src = frame['src_ids'].tolist()
        self.tgt = frame['tgt_ids'].tolist()
    def __len__(self): return len(self.src)
    def __getitem__(self, i):
        return {'src': torch.tensor(self.src[i], dtype=torch.long),
                'tgt': torch.tensor(self.tgt[i], dtype=torch.long)}

def collate_fn(batch):
    srcs=[b['src'] for b in batch]; tgts=[b['tgt'] for b in batch]
    ms=max(len(s) for s in srcs); mt=max(len(t) for t in tgts)
    ps=torch.full((len(batch),ms),PAD_ID,dtype=torch.long)
    pt=torch.full((len(batch),mt),PAD_ID,dtype=torch.long)
    for i,s in enumerate(srcs): ps[i,:len(s)]=s
    for i,t in enumerate(tgts): pt[i,:len(t)]=t
    return {'src': ps, 'tgt': pt}

BATCH_SIZE = 32
train_loader = DataLoader(SummarizationDataset(train_df), batch_size=BATCH_SIZE,
                          shuffle=True, collate_fn=collate_fn, num_workers=2, pin_memory=True)
val_loader = DataLoader(SummarizationDataset(val_df), batch_size=BATCH_SIZE,
                        shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=True)

D_MODEL, NHEAD = 256, 8
NUM_ENC, NUM_DEC = 4, 4
DIM_FF, DROPOUT, MAX_POS = 1024, 0.2, 512

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=MAX_POS, dropout=DROPOUT):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).float().unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() *
                        (-math.log(10000.0)/d_model))
        pe[:,0::2]=torch.sin(pos*div); pe[:,1::2]=torch.cos(pos*div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1), :])

class TransformerSummarizer(nn.Module):
    def __init__(self):
        super().__init__()
        self.d_model = D_MODEL
        self.embedding = nn.Embedding(VOCAB_SIZE, D_MODEL, padding_idx=PAD_ID)
        self.pos_encoder = PositionalEncoding(D_MODEL)
        self.transformer = nn.Transformer(
            d_model=D_MODEL, nhead=NHEAD,
            num_encoder_layers=NUM_ENC, num_decoder_layers=NUM_DEC,
            dim_feedforward=DIM_FF, dropout=DROPOUT,
            activation='gelu', batch_first=True, norm_first=False)
        self.fc_out = nn.Linear(D_MODEL, VOCAB_SIZE, bias=False)
        self.fc_out.weight = self.embedding.weight
        for p in self.parameters():
            if p.dim() > 1: nn.init.xavier_uniform_(p)
    def forward(self, src, tgt, tgt_mask=None,
                src_key_padding_mask=None, tgt_key_padding_mask=None):
        se = self.pos_encoder(self.embedding(src)*math.sqrt(self.d_model))
        te = self.pos_encoder(self.embedding(tgt)*math.sqrt(self.d_model))
        if tgt_mask is None:
            tgt_mask = nn.Transformer.generate_square_subsequent_mask(
                tgt.size(1), device=src.device)
        out = self.transformer(se, te, tgt_mask=tgt_mask,
            src_key_padding_mask=src_key_padding_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=src_key_padding_mask)
        return self.fc_out(out)

model = TransformerSummarizer().to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")

criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID, label_smoothing=0.1)
LR_PEAK, WARMUP = 3e-4, 2000
optimizer = torch.optim.AdamW(model.parameters(), lr=LR_PEAK,
                              betas=(0.9,0.98), weight_decay=0.01)
def lr_lambda(step):
    step = max(step, 1)
    if step < WARMUP: return step / WARMUP
    return math.sqrt(WARMUP / step)
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# --- Training ---
EPOCHS, CLIP, PATIENCE = 40, 1.0, 8
scaler = torch.cuda.amp.GradScaler()
best_val = float('inf'); no_improve = 0
history = {'train': [], 'val': []}

@torch.no_grad()
def validate():
    model.eval(); tot, ntok = 0.0, 0
    for batch in val_loader:
        src = batch['src'].to(device); tgt = batch['tgt'].to(device)
        ti, to = tgt[:,:-1], tgt[:,1:]
        pad = (src==PAD_ID); tpad = (ti==PAD_ID)
        causal = nn.Transformer.generate_square_subsequent_mask(ti.size(1), device=device)
        with torch.cuda.amp.autocast():
            logits = model(src, ti, tgt_mask=causal,
                           src_key_padding_mask=pad, tgt_key_padding_mask=tpad)
            loss = criterion(logits.reshape(-1,VOCAB_SIZE), to.reshape(-1))
        nt = (to!=PAD_ID).sum().item(); tot += loss.item()*nt; ntok += nt
    return tot/max(ntok,1)

for epoch in range(1, EPOCHS+1):
    model.train(); running, ntok = 0.0, 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch:02d}", leave=False)
    for batch in pbar:
        src = batch['src'].to(device); tgt = batch['tgt'].to(device)
        ti, to = tgt[:,:-1], tgt[:,1:]
        pad = (src==PAD_ID); tpad = (ti==PAD_ID)
        causal = nn.Transformer.generate_square_subsequent_mask(ti.size(1), device=device)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast():
            logits = model(src, ti, tgt_mask=causal,
                           src_key_padding_mask=pad, tgt_key_padding_mask=tpad)
            loss = criterion(logits.reshape(-1,VOCAB_SIZE), to.reshape(-1))
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP)
        scaler.step(optimizer); scaler.update(); scheduler.step()
        nt = (to!=PAD_ID).sum().item(); running += loss.item()*nt; ntok += nt
        pbar.set_postfix({'loss':f"{running/ntok:.4f}"})

    train_loss = running/max(ntok,1); val_loss = validate()
    history['train'].append(train_loss); history['val'].append(val_loss)
    print(f"Epoch {epoch:02d} | train {train_loss:.4f} | val {val_loss:.4f}")

    if val_loss < best_val - 1e-4:
        best_val = val_loss; no_improve = 0
        torch.save({'model_state': model.state_dict(),
                    'epoch': epoch, 'val_loss': val_loss}, 'transformer_best.pt')
        print("  -> saved best")
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print("Early stopping."); break

with open('history.json','w') as f: json.dump(history, f)
print(f"\nBest val loss: {best_val:.4f}")

# Verify persistence
print("\n=== /kaggle/working files ===")
for fn in ["nepali_sp.model", "splits.pkl", "transformer_best.pt", "history.json"]:
    p = os.path.join('/kaggle/working', fn)
    if os.path.exists(p):
        print(f"  ✓ {fn}  ({os.path.getsize(p)/1e6:.2f} MB)")
    else:
        print(f"  ✗ {fn}  MISSING!")

Device: cuda
Parameters: 15.57M


/tmp/ipykernel_23/1840072254.py:96: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


Epoch 01:   0%|          | 0/1248 [00:00<?, ?it/s]

/tmp/ipykernel_23/1840072254.py:124: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/usr/local/lib/python3.13/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(
/tmp/ipykernel_23/1840072254.py:108: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 01 | train 8.5033 | val 7.3505
  -> saved best


Epoch 02:   0%|          | 0/1248 [00:00<?, ?it/s]

... (73 outputs left out)

Epoch 39:   0%|          | 0/1248 [00:00<?, ?it/s]

Epoch 39 | train 3.5446 | val 3.8865
  -> saved best


Epoch 40:   0%|          | 0/1248 [00:00<?, ?it/s]

Epoch 40 | train 3.5292 | val 3.8916

Best val loss: 3.8865

=== /kaggle/working files ===
  ✓ nepali_sp.model  (1.17 MB)
  ✓ splits.pkl  (342.76 MB)
  ✓ transformer_best.pt  (62.84 MB)
  ✓ history.json  (0.00 MB)


In [3]:
# ============================================================
# CELL 3 (FIXED): Inference with beam search
# ============================================================
import os, math, pickle
import torch, torch.nn as nn
import pandas as pd
import sentencepiece as spm
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

os.chdir('/kaggle/working')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

sp = spm.SentencePieceProcessor(); sp.load("nepali_sp.model")
PAD_ID, BOS_ID, EOS_ID = sp.pad_id(), sp.bos_id(), sp.eos_id()
VOCAB_SIZE = sp.get_piece_size()

with open("splits.pkl", "rb") as f:
    splits = pickle.load(f)
test_df = splits['test']
print(f"Test rows: {len(test_df)}")

MAX_TGT_LEN = 64
class SummarizationDataset(torch.utils.data.Dataset):
    def __init__(self, frame):
        self.src = frame['src_ids'].tolist()
        self.tgt = frame['tgt_ids'].tolist()
    def __len__(self): return len(self.src)
    def __getitem__(self, i):
        return {'src': torch.tensor(self.src[i], dtype=torch.long),
                'tgt': torch.tensor(self.tgt[i], dtype=torch.long)}

def collate_fn(batch):
    srcs=[b['src'] for b in batch]; tgts=[b['tgt'] for b in batch]
    ms=max(len(s) for s in srcs); mt=max(len(t) for t in tgts)
    ps=torch.full((len(batch),ms),PAD_ID,dtype=torch.long)
    pt=torch.full((len(batch),mt),PAD_ID,dtype=torch.long)
    for i,s in enumerate(srcs): ps[i,:len(s)]=s
    for i,t in enumerate(tgts): pt[i,:len(t)]=t
    return {'src': ps, 'tgt': pt}

D_MODEL, NHEAD, NUM_ENC, NUM_DEC, DIM_FF, DROPOUT, MAX_POS = 256, 8, 4, 4, 1024, 0.2, 512

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=MAX_POS, dropout=DROPOUT):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).float().unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0)/d_model))
        pe[:,0::2]=torch.sin(pos*div); pe[:,1::2]=torch.cos(pos*div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x): return self.dropout(x + self.pe[:, :x.size(1), :])

class TransformerSummarizer(nn.Module):
    def __init__(self):
        super().__init__()
        self.d_model = D_MODEL
        self.embedding = nn.Embedding(VOCAB_SIZE, D_MODEL, padding_idx=PAD_ID)
        self.pos_encoder = PositionalEncoding(D_MODEL)
        self.transformer = nn.Transformer(
            d_model=D_MODEL, nhead=NHEAD,
            num_encoder_layers=NUM_ENC, num_decoder_layers=NUM_DEC,
            dim_feedforward=DIM_FF, dropout=DROPOUT,
            activation='gelu', batch_first=True, norm_first=False)
        self.fc_out = nn.Linear(D_MODEL, VOCAB_SIZE, bias=False)
        self.fc_out.weight = self.embedding.weight
    def forward(self, src, tgt, tgt_mask=None,
                src_key_padding_mask=None, tgt_key_padding_mask=None):
        se = self.pos_encoder(self.embedding(src)*math.sqrt(self.d_model))
        te = self.pos_encoder(self.embedding(tgt)*math.sqrt(self.d_model))
        if tgt_mask is None:
            tgt_mask = nn.Transformer.generate_square_subsequent_mask(
                tgt.size(1), device=src.device)
        out = self.transformer(se, te, tgt_mask=tgt_mask,
            src_key_padding_mask=src_key_padding_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=src_key_padding_mask)
        return self.fc_out(out)

model = TransformerSummarizer().to(device)
ckpt = torch.load('transformer_best.pt', map_location=device)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f"Loaded epoch={ckpt['epoch']}, val={ckpt['val_loss']:.4f}")

# ============================================================
# BEAM SEARCH — FIXED reordering logic
# ============================================================
@torch.no_grad()
def beam_generate(src_batch, num_beams=4, max_new=MAX_TGT_LEN):
    B = src_batch.size(0)
    src_pad = (src_batch == PAD_ID)
    src_batch = src_batch.repeat_interleave(num_beams, dim=0)   # (B*num_beams, L)
    src_pad   = src_pad.repeat_interleave(num_beams, dim=0)

    # tgt: (B*num_beams, 1), all BOS
    tgt = torch.full((B*num_beams, 1), BOS_ID, dtype=torch.long, device=device)

    # Beam scores: (B, num_beams), only beam 0 starts at 0
    scores = torch.zeros(B, num_beams, device=device)
    scores[:, 1:] = -1e9
    scores = scores.view(-1)                                    # (B*num_beams,)

    # Track finished beams (optional — keeps highest-scoring non-EOS path)
    finished = torch.zeros(B, dtype=torch.bool, device=device)

    for _ in range(max_new):
        L = tgt.size(1)
        causal = nn.Transformer.generate_square_subsequent_mask(L, device=device)
        with torch.cuda.amp.autocast():
            logits = model(src_batch, tgt, tgt_mask=causal,
                           src_key_padding_mask=src_pad,
                           tgt_key_padding_mask=(tgt == PAD_ID))
        logp = torch.log_softmax(logits[:, -1, :], dim=-1)       # (B*num_beams, V)

        # Combine running score + next-token log-prob
        cand = scores.unsqueeze(1) + logp                        # (B*num_beams, V)
        cand = cand.view(B, num_beams * VOCAB_SIZE)              # (B, num_beams*V)

        topk_scores, topk_ids = cand.topk(num_beams, dim=-1)     # (B, num_beams)
        sel_scores = topk_scores.reshape(-1)                     # (B*num_beams,)
        sel_ids    = topk_ids.reshape(-1)                        # (B*num_beams,)

        beam_idx = sel_ids // VOCAB_SIZE                         # (B*num_beams,)
        tok_idx  = sel_ids % VOCAB_SIZE                          # (B*num_beams,)

        # ---- CORRECT reordering using torch.gather ----
        tgt_3d = tgt.view(B, num_beams, -1)                      # (B, num_beams, L)
        gather_idx = beam_idx.view(B, num_beams, 1).expand(-1, -1, tgt_3d.size(-1))
        tgt_3d = torch.gather(tgt_3d, 1, gather_idx)             # (B, num_beams, L)
        tgt = tgt_3d.reshape(B * num_beams, -1)                  # (B*num_beams, L)
        tgt = torch.cat([tgt, tok_idx.unsqueeze(1)], dim=1)      # (B*num_beams, L+1)

        scores = sel_scores                                      # (B*num_beams,)

        # Optional: stop early if all beams per example ended with EOS
        last_tok = tok_idx.view(B, num_beams)                    # (B, num_beams)
        just_finished = (last_tok == EOS_ID).all(dim=1)          # (B,)
        finished |= just_finished
        if finished.all():
            break

    # Pick best beam per example
    tgt = tgt.view(B, num_beams, -1)                             # (B, num_beams, L)
    best = scores.view(B, num_beams).argmax(dim=-1)              # (B,)
    chosen = tgt[torch.arange(B, device=device), best, :]        # (B, L)
    return chosen

def decode_row(ids):
    ids = [t for t in ids if t not in (PAD_ID, BOS_ID, EOS_ID)]
    return sp.decode(ids)

# ---------- Quick smoke test on 2 examples BEFORE full run ----------
print("Smoke test on 2 examples...")
smoke_loader = DataLoader(SummarizationDataset(test_df.head(2)),
                          batch_size=2, shuffle=False, collate_fn=collate_fn)
for b in smoke_loader:
    gen = beam_generate(b['src'].to(device), num_beams=4)
    print(f"  gen shape: {tuple(gen.shape)}")   # should be (2, <=64)
    for row in gen.cpu().tolist():
        print("  pred:", decode_row(row)[:100])
print("Smoke test passed.\n")

# ---------- Full run ----------
test_loader = DataLoader(SummarizationDataset(test_df), batch_size=16,
                         shuffle=False, collate_fn=collate_fn)

records = []
for batch in tqdm(test_loader, desc="Generating"):
    gen = beam_generate(batch['src'].to(device), num_beams=4)
    for row in gen.cpu().tolist():
        records.append(decode_row(row))

pred_df = pd.DataFrame({
    'source':    test_df['content'].astype(str).tolist(),
    'reference': test_df['summary'].astype(str).tolist(),
    'predicted': records,
})
pred_df.to_csv('test_predictions.csv', index=False, encoding='utf-8')
print(f"\nSaved {len(pred_df)} predictions to test_predictions.csv")
print(pred_df.head(2).to_string(max_colwidth=80))

Test rows: 4992
Loaded epoch=39, val=3.8865
Smoke test on 2 examples...


/tmp/ipykernel_23/2522507900.py:111: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/usr/local/lib/python3.13/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(


  gen shape: (2, 64)
  pred: कुलमान घिसिङले जलविद्युत् क्षेत्रको विकास सम्भव भएको भन्दै आईपीओ नरोक्न सम्बद्ध निकायको ध्यानाकर्षण 
  pred: त्रिभुवन विश्वविद्यालयले स्वास्थ्य सुरक्षा मापदण्ड लागू गरी विद्यार्थीलाई क्याम्पसमै उपस्थित गराएर क
Smoke test passed.



Generating:   0%|          | 0/312 [00:00<?, ?it/s]


Saved 4992 predictions to test_predictions.csv
                                                                            source                                                                        reference                                                                        predicted
0  सर्वसाधारणको लगानीबाटै जलविद्युत् क्षेत्रको विकास सम्भव भएको भन्दै घिसिङले आ...  जलविद्युत् आईपीओ नरोक्न कुलमानको आग्रह। सर्वसाधारणको लगानीबाटै जलविद्युत् क्...  कुलमान घिसिङले जलविद्युत् क्षेत्रको विकास सम्भव भएको भन्दै आईपीओ नरोक्न सम्ब...
1  त्रिभुवन विश्वविद्यालयले स्वास्थ्य सुरक्षा सम्बन्धी मापदण्ड लागू गरेर सकेसम्...  त्रिभुवन विश्वविद्यालयले स्वास्थ्य सुरक्षा मापदण्ड पालना गरी विद्यार्थीलाई भ...  त्रिभुवन विश्वविद्यालयले स्वास्थ्य सुरक्षा मापदण्ड लागू गरी विद्यार्थीलाई क्...


In [4]:
# ============================================================
# CELL 4: Evaluation
# ============================================================
!pip install -q sacrebleu rouge-score bert-score
import pandas as pd, numpy as np
from sacrebleu import corpus_bleu
from rouge_score import rouge_scorer
from bert_score import score as bert_score_fn

pred_df = pd.read_csv('/kaggle/working/test_predictions.csv')
refs = pred_df['reference'].astype(str).tolist()
hyps = pred_df['predicted'].astype(str).tolist()

bleu = corpus_bleu(hyps, [refs])
print(f"SacreBLEU: {bleu.score:.4f}")

scorer = rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=False)
r1,r2,rL = [],[],[]
for h,r in zip(hyps,refs):
    s = scorer.score(r,h)
    r1.append(s['rouge1'].fmeasure); r2.append(s['rouge2'].fmeasure); rL.append(s['rougeL'].fmeasure)
print(f"ROUGE-1: {np.mean(r1):.4f}")
print(f"ROUGE-2: {np.mean(r2):.4f}")
print(f"ROUGE-L: {np.mean(rL):.4f}")

P,R,F1 = bert_score_fn(hyps, refs, lang='ne', verbose=False)
print(f"BERTScore P: {P.mean().item():.4f}")
print(f"BERTScore R: {R.mean().item():.4f}")
print(f"BERTScore F1: {F1.mean().item():.4f}")

def jac(a,b,n=2):
    a,b=str(a),str(b)
    A=set(a[i:i+n] for i in range(max(len(a)-n+1,0)))
    B=set(b[i:i+n] for i in range(max(len(b)-n+1,0)))
    if not A and not B: return 1.0
    if not A or not B: return 0.0
    return len(A&B)/len(A|B)
print(f"Jaccard: {np.mean([jac(h,r) for h,r in zip(hyps,refs)]):.4f}")

SacreBLEU: 19.8632
ROUGE-1: 0.0027
ROUGE-2: 0.0024
ROUGE-L: 0.0027


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  714MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERTScore P: 0.7970
BERTScore R: 0.7818
BERTScore F1: 0.7888
Jaccard: 0.5368


In [5]:
# ============================================================
# FINAL EVALUATION — prints all 6 scores
# ============================================================
!pip install -q sacrebleu rouge-score bert-score

import pandas as pd
import numpy as np
from sacrebleu import corpus_bleu
from rouge_score import rouge_scorer
from bert_score import score as bert_score_fn
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# ---------- Load predictions ----------
pred_df = pd.read_csv('/kaggle/working/test_predictions.csv')
refs = pred_df['reference'].astype(str).tolist()
hyps = pred_df['predicted'].astype(str).tolist()
print(f"Evaluating {len(pred_df)} predictions\n")

# ---------- 1. SacreBLEU ----------
bleu = corpus_bleu(hyps, [refs]).score

# ---------- 2–4. ROUGE ----------
scorer = rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=False)
r1, r2, rL = [], [], []
for h, r in zip(hyps, refs):
    s = scorer.score(r, h)
    r1.append(s['rouge1'].fmeasure)
    r2.append(s['rouge2'].fmeasure)
    rL.append(s['rougeL'].fmeasure)
rouge1 = float(np.mean(r1))
rouge2 = float(np.mean(r2))
rougeL = float(np.mean(rL))

# ---------- 5. BERTScore ----------
P, R, F1 = bert_score_fn(hyps, refs, lang='ne', verbose=False, device=device)
bs_p  = P.mean().item()
bs_r  = R.mean().item()
bs_f1 = F1.mean().item()

# ---------- 6. Jaccard (char bigram) ----------
def jac(a, b, n=2):
    a, b = str(a), str(b)
    A = set(a[i:i+n] for i in range(max(len(a)-n+1, 0)))
    B = set(b[i:i+n] for i in range(max(len(b)-n+1, 0)))
    if not A and not B: return 1.0
    if not A or not B: return 0.0
    return len(A & B) / len(A | B)
jaccard = float(np.mean([jac(h, r) for h, r in zip(hyps, refs)]))

# ---------- Print all scores ----------
print("=" * 50)
print("FINAL TRANSFORMER RESULTS")
print("=" * 50)
print(f"SacreBLEU               : {bleu:.4f}")
print(f"ROUGE-1                 : {rouge1:.4f}")
print(f"ROUGE-2                 : {rouge2:.4f}")
print(f"ROUGE-L                 : {rougeL:.4f}")
print(f"BERTScore Precision     : {bs_p:.4f}")
print(f"BERTScore Recall        : {bs_r:.4f}")
print(f"BERTScore F1            : {bs_f1:.4f}")
print(f"Jaccard (char bigram)   : {jaccard:.4f}")
print("=" * 50)

# ---------- Extra sanity checks ----------
empty = sum(1 for h in hyps if len(h.strip()) == 0)
print(f"\nEmpty predictions       : {empty} / {len(hyps)}")
print(f"Avg prediction length   : {np.mean([len(h) for h in hyps]):.1f} chars")
print(f"Avg reference length    : {np.mean([len(r) for r in refs]):.1f} chars")

# ---------- Save results to file ----------
results = {
    'sacrebleu': round(bleu, 4),
    'rouge1': round(rouge1, 4),
    'rouge2': round(rouge2, 4),
    'rougeL': round(rougeL, 4),
    'bertscore_p': round(bs_p, 4),
    'bertscore_r': round(bs_r, 4),
    'bertscore_f1': round(bs_f1, 4),
    'jaccard': round(jaccard, 4),
}
pd.Series(results).to_csv('/kaggle/working/final_scores.csv')
print("\nSaved final_scores.csv")

Evaluating 4992 predictions



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FINAL TRANSFORMER RESULTS
SacreBLEU               : 19.8632
ROUGE-1                 : 0.0027
ROUGE-2                 : 0.0024
ROUGE-L                 : 0.0027
BERTScore Precision     : 0.7970
BERTScore Recall        : 0.7818
BERTScore F1            : 0.7888
Jaccard (char bigram)   : 0.5368

Empty predictions       : 0 / 4992
Avg prediction length   : 291.8 chars
Avg reference length    : 300.0 chars

Saved final_scores.csv


In [6]:
import pandas as pd
pred_df = pd.read_csv('/kaggle/working/test_predictions.csv')

for i in range(5):
    print(f"\n===== Sample {i+1} =====")
    print(f"REF : {pred_df['reference'].iloc[i][:250]}")
    print(f"PRED: {pred_df['predicted'].iloc[i][:250]}")


===== Sample 1 =====
REF : जलविद्युत् आईपीओ नरोक्न कुलमानको आग्रह। सर्वसाधारणको लगानीबाटै जलविद्युत् क्षेत्रको विकास सम्भव भएको भन्दै घिसिङले आईपीओ नरोक्न सम्बद्ध निकायको ध्यानाकर्षण गराएका हुन्
PRED: कुलमान घिसिङले जलविद्युत् क्षेत्रको विकास सम्भव भएको भन्दै आईपीओ नरोक्न सम्बद्ध निकायको ध्यानाकर्षण गराएका छन्। नेपाल विद्युत प्राधिकरणका कार्यकारी निर्देशक कुलमान घिसिङले आईपीओ नरोक्न र जलविद्युत क्षेत्रको विकास सम्भव भएको भन्दै आईपीओ नरोक्न आग्रह ग

===== Sample 2 =====
REF : त्रिभुवन विश्वविद्यालयले स्वास्थ्य सुरक्षा मापदण्ड पालना गरी विद्यार्थीलाई भौतिक रूपमै उपस्थित गराएर कक्षा सञ्चालन गर्न निर्देशन दिएको छ। स्थानीय तहसँग समन्वय गरी पठनपाठन सुचारु गर्न त्रिविले आग्रह गरेको छ। भौतिक उपस्थिति सम्भव नभए वैकल्पिक विधि अपना
PRED: त्रिभुवन विश्वविद्यालयले स्वास्थ्य सुरक्षा मापदण्ड लागू गरी विद्यार्थीलाई क्याम्पसमै उपस्थित गराएर कक्षा सञ्चालन गर्न निर्देशन दिएको छ। तोकिएको समय सीमा र मापदण्डभित्र रहने गरी स्थानीय तहसँग समन्वय गरी पठनपाठन सुचारु गर्न आग्रह गरिएको छ। त्रिभुवन विश

===== Samp

In [7]:
# ============================================================
# ROUGE — manual implementation (bypasses rouge_score bug)
# ============================================================
import pandas as pd, numpy as np
from collections import Counter

pred_df = pd.read_csv('/kaggle/working/test_predictions.csv')
refs = pred_df['reference'].astype(str).tolist()
hyps = pred_df['predicted'].astype(str).tolist()

def ngrams(tokens, n):
    return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

def rouge_n_f(pred_tokens, ref_tokens, n):
    p_ng = ngrams(pred_tokens, n)
    r_ng = ngrams(ref_tokens, n)
    if not p_ng or not r_ng:
        return 0.0
    overlap = sum((Counter(p_ng) & Counter(r_ng)).values())
    p = overlap / len(p_ng)
    r = overlap / len(r_ng)
    return 0.0 if p + r == 0 else 2 * p * r / (p + r)

def rouge_l_f(pred_tokens, ref_tokens):
    m, n = len(pred_tokens), len(ref_tokens)
    if m == 0 or n == 0:
        return 0.0
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(1, m+1):
        for j in range(1, n+1):
            if pred_tokens[i-1] == ref_tokens[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])
    lcs = dp[m][n]
    p = lcs / m
    r = lcs / n
    return 0.0 if p + r == 0 else 2 * p * r / (p + r)

r1, r2, rL = [], [], []
for h, r in zip(hyps, refs):
    ht, rt = h.split(), r.split()
    r1.append(rouge_n_f(ht, rt, 1))
    r2.append(rouge_n_f(ht, rt, 2))
    rL.append(rouge_l_f(ht, rt))

print(f"ROUGE-1: {np.mean(r1):.4f}")
print(f"ROUGE-2: {np.mean(r2):.4f}")
print(f"ROUGE-L: {np.mean(rL):.4f}")

ROUGE-1: 0.4185
ROUGE-2: 0.2448
ROUGE-L: 0.3594


In [8]:
# ============================================================
# JACCARD — multiple legitimate variants (FIXED)
# ============================================================
import pandas as pd
import numpy as np
from collections import Counter

pred_df = pd.read_csv('/kaggle/working/test_predictions.csv')
refs = pred_df['reference'].astype(str).tolist()
hyps = pred_df['predicted'].astype(str).tolist()

def set_jaccard(A, B):
    if not A and not B: return 1.0
    if not A or not B:  return 0.0
    return len(A & B) / len(A | B)

def multiset_jaccard(A, B):
    if not A and not B: return 1.0
    if not A or not B:  return 0.0
    keys = set(A) | set(B)
    num = sum(min(A.get(k, 0), B.get(k, 0)) for k in keys)
    den = sum(max(A.get(k, 0), B.get(k, 0)) for k in keys)
    return 0.0 if den == 0 else num / den

def jac_char_set(a, b, n=2):
    a, b = str(a), str(b)
    A = set(a[i:i+n] for i in range(max(len(a)-n+1, 0)))
    B = set(b[i:i+n] for i in range(max(len(b)-n+1, 0)))
    return set_jaccard(A, B)

def jac_word_set(a, b):
    return set_jaccard(set(str(a).split()), set(str(b).split()))

def jac_word_bigram_set(a, b):
    ta, tb = str(a).split(), str(b).split()
    A = set((ta[i], ta[i+1]) for i in range(len(ta)-1))
    B = set((tb[i], tb[i+1]) for i in range(len(tb)-1))
    return set_jaccard(A, B)

def jac_word_mset(a, b):
    return multiset_jaccard(Counter(str(a).split()), Counter(str(b).split()))

def jac_char_mset(a, b, n=2):
    a, b = str(a), str(b)
    A = Counter(a[i:i+n] for i in range(max(len(a)-n+1, 0)))
    B = Counter(b[i:i+n] for i in range(max(len(b)-n+1, 0)))
    return multiset_jaccard(A, B)

# ---------- All variants wrapped in lambdas ----------
variants = {
    'Char-2gram  (set)':      lambda a, b: jac_char_set(a, b, n=2),
    'Char-3gram  (set)':      lambda a, b: jac_char_set(a, b, n=3),
    'Char-2gram  (multiset)': lambda a, b: jac_char_mset(a, b, n=2),
    'Word-unigram(set)':      jac_word_set,
    'Word-bigram (set)':      jac_word_bigram_set,
    'Word-unigram(multiset)': jac_word_mset,
}

results = {}
for name, fn in variants.items():
    vals = [fn(h, r) for h, r in zip(hyps, refs)]
    results[name] = (float(np.mean(vals)), float(np.std(vals)))

print("=" * 58)
print("  JACCARD SIMILARITY — VARIANTS")
print("=" * 58)
print(f"  {'Variant':<28} {'Mean':>8} {'Std':>8}")
print("-" * 58)
for name, (mean, std) in results.items():
    print(f"  {name:<28} {mean:>8.4f} {std:>8.4f}")
print("=" * 58)

pd.DataFrame(
    [(k, round(v[0], 4), round(v[1], 4)) for k, v in results.items()],
    columns=['variant', 'mean', 'std']
).to_csv('/kaggle/working/jaccard_variants.csv', index=False)
print("\nSaved jaccard_variants.csv")

  JACCARD SIMILARITY — VARIANTS
  Variant                          Mean      Std
----------------------------------------------------------
  Char-2gram  (set)              0.5368   0.1141
  Char-3gram  (set)              0.3955   0.1326
  Char-2gram  (multiset)         0.4675   0.0970
  Word-unigram(set)              0.3101   0.1396
  Word-bigram (set)              0.1627   0.1283
  Word-unigram(multiset)         0.2748   0.1178

Saved jaccard_variants.csv
